In [17]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
project_root = next((p for p in [cwd, *cwd.parents] if (p / "src").exists()), None)
if project_root is None:
    raise RuntimeError("No se encontro la raiz del proyecto (carpeta 'src').")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.geospatial_expansion import agregar_distancias_minimas_poi

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

In [18]:
DATA_DIR = Path("../data/processed/idealista/sale_homes_run_20260218_173035")
DATA_IDEALISTA = DATA_DIR / "sale_homes_cantabria_bezana_like_raw.csv"

df = pd.read_csv(DATA_IDEALISTA)

df.head()

,propertyCode,thumbnail,externalReference,numPhotos,floor,price,propertyType,operation,size,exterior,rooms,bathrooms,address,province,municipality,district,country,latitude,longitude,showAddress,url,distance,description,hasVideo,status,newDevelopment,hasLift,priceByArea,hasPlan,has3DTour,has360,hasStaging,notes,topNewDevelopment,newDevelopmentHighlight,topPlus,priceInfo.price.amount,priceInfo.price.currencySuffix,parkingSpace.hasParkingSpace,parkingSpace.isParkingSpaceIncludedInPrice,detailedType.typology,suggestedTexts.subtitle,suggestedTexts.title,detailedType.subTypology,highlight.groupDescription,newDevelopmentFinished,parkingSpace.parkingSpacePrice,priceInfo.price.priceDropInfo.formerPrice,priceInfo.price.priceDropInfo.priceDropValue,priceInfo.price.priceDropInfo.priceDropPercentage
0,110673583,https://img4.idealista.com/blur/480_360_mq/0/i...,40131,27,1,375000.0,flat,sale,79.0,True,2,1,calle la Pereda,Cantabria,Santander,Valdenoja,es,43.480623,-3.797372,False,https://www.idealista.com/inmueble/110673583/,9514,"Se vende piso de 2 habitaciones, salón, cocina...",False,good,False,True,4747.0,False,False,False,False,[],False,False,False,375000.0,€,True,True,flat,"Valdenoja, Santander",Piso en calle la Pereda,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,110673566,https://img4.idealista.com/blur/480_360_mq/0/i...,40134,43,NaN,500000.0,chalet,sale,223.0,NaN,4,3,Barrio el Hondal,Cantabria,Polanco,NaN,es,43.387446,-4.017180,False,https://www.idealista.com/inmueble/110673566/,11090,Se vende casa pareada esquinera con 350 metros...,False,good,False,NaN,2242.0,False,False,False,False,[],False,False,False,500000.0,€,True,True,chalet,Polanco,Chalet pareado en Barrio el Hondal,semidetachedHouse,NaN,NaN,NaN,NaN,NaN,NaN
2,110672622,https://img4.idealista.com/blur/480_360_mq/0/i...,ATRV1237,22,5,190000.0,flat,sale,56.0,True,1,1,"calle las Cagigas, 2",Cantabria,Santander,Alisal - Cazoña - San Román,es,43.456979,-3.854764,True,https://www.idealista.com/inmueble/110672622/,4217,Se pone a la venta un amplio apartamento en la...,False,good,False,True,3393.0,False,False,False,False,[],False,False,False,190000.0,€,NaN,NaN,flat,"Alisal - Cazoña - San Román, Santander","Piso en calle las Cagigas, 2",NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,110672483,https://img4.idealista.com/blur/480_360_mq/0/i...,86086,11,NaN,169000.0,studio,sale,45.0,True,0,1,calle San Fernando,Cantabria,Santander,Numancia - San Fernando,es,43.458724,-3.821445,False,https://www.idealista.com/inmueble/110672483/,6844,"Magnífico estudio en venta en Santander, ubica...",False,good,False,True,3756.0,False,False,False,False,[],False,False,False,169000.0,€,NaN,NaN,flat,"Numancia - San Fernando, Santander",Estudio en calle San Fernando,studio,NaN,NaN,NaN,NaN,NaN,NaN
4,110663429,https://img4.idealista.com/blur/480_360_mq/0/i...,04607,15,3,192000.0,flat,sale,58.0,True,2,1,Cuatro Caminos,Cantabria,Santander,Cuatro Caminos,es,43.460521,-3.827775,False,https://www.idealista.com/inmueble/110663429/,6406,Junto a la Plaza de Cuatro caminos y a pocos m...,False,good,False,False,3310.0,False,False,False,False,[],False,False,False,192000.0,€,NaN,NaN,flat,"Cuatro Caminos, Santander",Piso,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
print(f"Columnas totales: {df.columns}")

columns = ['propertyCode','price','size','rooms','bathrooms','detailedType.typology','detailedType.subTypology',
                'province','municipality','district','latitude','longitude','floor','exterior','hasLift',
                'parkingSpace.hasParkingSpace','newDevelopment']

df_reduced = df[columns].copy()
df_reduced.head()

Columnas totales: Index(['propertyCode', 'thumbnail', 'externalReference', 'numPhotos', 'floor', 'price', 'propertyType', 'operation', 'size', 'exterior',
       'rooms', 'bathrooms', 'address', 'province', 'municipality', 'district', 'country', 'latitude', 'longitude', 'showAddress', 'url',
       'distance', 'description', 'hasVideo', 'status', 'newDevelopment', 'hasLift', 'priceByArea', 'hasPlan', 'has3DTour', 'has360',
       'hasStaging', 'notes', 'topNewDevelopment', 'newDevelopmentHighlight', 'topPlus', 'priceInfo.price.amount',
       'priceInfo.price.currencySuffix', 'parkingSpace.hasParkingSpace', 'parkingSpace.isParkingSpaceIncludedInPrice',
       'detailedType.typology', 'suggestedTexts.subtitle', 'suggestedTexts.title', 'detailedType.subTypology',
       'highlight.groupDescription', 'newDevelopmentFinished', 'parkingSpace.parkingSpacePrice',
       'priceInfo.price.priceDropInfo.formerPrice', 'priceInfo.price.priceDropInfo.priceDropValue',
       'priceInfo.price.priceDr

,propertyCode,price,size,rooms,bathrooms,detailedType.typology,detailedType.subTypology,province,municipality,district,latitude,longitude,floor,exterior,hasLift,parkingSpace.hasParkingSpace,newDevelopment
0,110673583,375000.0,79.0,2,1,flat,NaN,Cantabria,Santander,Valdenoja,43.480623,-3.797372,1,True,True,True,False
1,110673566,500000.0,223.0,4,3,chalet,semidetachedHouse,Cantabria,Polanco,NaN,43.387446,-4.017180,NaN,NaN,NaN,True,False
2,110672622,190000.0,56.0,1,1,flat,NaN,Cantabria,Santander,Alisal - Cazoña - San Román,43.456979,-3.854764,5,True,True,NaN,False
3,110672483,169000.0,45.0,0,1,flat,studio,Cantabria,Santander,Numancia - San Fernando,43.458724,-3.821445,NaN,True,True,NaN,False
4,110663429,192000.0,58.0,2,1,flat,NaN,Cantabria,Santander,Cuatro Caminos,43.460521,-3.827775,3,True,False,NaN,False


In [20]:
columnas_espanol = {
    "propertyCode": "codigo_inmueble",
    "price": "precio",
    "size": "superficie_construida_m2",
    "rooms": "numero_dormitorios",
    "bathrooms": "numero_banos",
    "detailedType.typology": "tipologia",
    "detailedType.subTypology": "subtipologia",
    "operation": "tipo_operacion",
    "province": "provincia",
    "municipality": "municipio",
    "district": "distrito",
    "latitude": "latitud",
    "longitude": "longitud",
    "floor": "planta",
    "exterior": "es_exterior",
    "hasLift": "tiene_ascensor",
    "parkingSpace.hasParkingSpace": "tiene_garaje",
    "newDevelopment": "obra_nueva"
}

df_es = df_reduced.rename(columns = columnas_espanol).copy()
df_es.head()

,codigo_inmueble,precio,superficie_construida_m2,numero_dormitorios,numero_banos,tipologia,subtipologia,provincia,municipio,distrito,latitud,longitud,planta,es_exterior,tiene_ascensor,tiene_garaje,obra_nueva
0,110673583,375000.0,79.0,2,1,flat,NaN,Cantabria,Santander,Valdenoja,43.480623,-3.797372,1,True,True,True,False
1,110673566,500000.0,223.0,4,3,chalet,semidetachedHouse,Cantabria,Polanco,NaN,43.387446,-4.017180,NaN,NaN,NaN,True,False
2,110672622,190000.0,56.0,1,1,flat,NaN,Cantabria,Santander,Alisal - Cazoña - San Román,43.456979,-3.854764,5,True,True,NaN,False
3,110672483,169000.0,45.0,0,1,flat,studio,Cantabria,Santander,Numancia - San Fernando,43.458724,-3.821445,NaN,True,True,NaN,False
4,110663429,192000.0,58.0,2,1,flat,NaN,Cantabria,Santander,Cuatro Caminos,43.460521,-3.827775,3,True,False,NaN,False


In [21]:
print(f"Total elementos duplicados: {df_es['codigo_inmueble'].duplicated().sum()} de {len(df_es)}")
df_es[df_es['codigo_inmueble'].duplicated(keep=False)].sort_values("codigo_inmueble").head()


Total elementos duplicados: 4346 de 4950


,codigo_inmueble,precio,superficie_construida_m2,numero_dormitorios,numero_banos,tipologia,subtipologia,provincia,municipio,distrito,latitud,longitud,planta,es_exterior,tiene_ascensor,tiene_garaje,obra_nueva
1781,94344476,124500.0,186.0,3,3,chalet,NaN,Cantabria,Polanco,NaN,43.39715,-4.008377,NaN,NaN,NaN,NaN,False
1483,94344476,124500.0,186.0,3,3,chalet,NaN,Cantabria,Polanco,NaN,43.39715,-4.008377,NaN,NaN,NaN,NaN,False
3263,94344476,124500.0,186.0,3,3,chalet,NaN,Cantabria,Polanco,NaN,43.39715,-4.008377,NaN,NaN,NaN,NaN,False
3372,94344476,124500.0,186.0,3,3,chalet,NaN,Cantabria,Polanco,NaN,43.39715,-4.008377,NaN,NaN,NaN,NaN,False
119,94344476,124500.0,186.0,3,3,chalet,NaN,Cantabria,Polanco,NaN,43.39715,-4.008377,NaN,NaN,NaN,NaN,False


In [22]:
df_clean = df_es.drop_duplicates(subset="codigo_inmueble", keep="first").copy()
print(f"Forma del nuevo dataset: {df_clean.shape}")


Forma del nuevo dataset: (604, 17)


In [23]:
df_clean['municipio'].value_counts().head(10)

municipio
Santander          130
Castro-Urdiales     77
Torrelavega         55
Noja                27
Santurtzi           27
Camargo             21
Piélagos            21
Laredo              20
Voto                16
Ortuella            16
Name: count, dtype: int64

In [24]:
df_expandido = agregar_distancias_minimas_poi(df_clean, ["playa", "supermercado"])

df_expandido.head()

,codigo_inmueble,precio,superficie_construida_m2,numero_dormitorios,numero_banos,tipologia,subtipologia,provincia,municipio,distrito,latitud,longitud,planta,es_exterior,tiene_ascensor,tiene_garaje,obra_nueva,distancia_min_playa_km,distancia_min_supermercado_km
0,110673583,375000.0,79.0,2,1,flat,NaN,Cantabria,Santander,Valdenoja,43.480623,-3.797372,1,True,True,True,False,0.902,0.253
1,110673566,500000.0,223.0,4,3,chalet,semidetachedHouse,Cantabria,Polanco,NaN,43.387446,-4.017180,NaN,NaN,NaN,True,False,4.699,0.609
2,110672622,190000.0,56.0,1,1,flat,NaN,Cantabria,Santander,Alisal - Cazoña - San Román,43.456979,-3.854764,5,True,True,NaN,False,2.774,0.254
3,110672483,169000.0,45.0,0,1,flat,studio,Cantabria,Santander,Numancia - San Fernando,43.458724,-3.821445,NaN,True,True,NaN,False,2.623,0.095
4,110663429,192000.0,58.0,2,1,flat,NaN,Cantabria,Santander,Cuatro Caminos,43.460521,-3.827775,3,True,False,NaN,False,2.239,0.160
